# 1. Imports 
Principais dependências:
1. atm_tc: importar pacote de funções;
2. Panmdas: Instalação via pip;

In [47]:
# Resolinção de root file
from pathlib import Path
import sys
import pandas as pd

In [48]:
# Define a raiz do projeto
ROOT = Path.cwd().parent  # ajuste conforme sua estrutura

# adiciona no PATH do Python
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(ROOT)

c:\Users\luiz.farias\OneDrive - Numerator International\Área de Trabalho\Workspace\PROD


In [49]:
def obter_periodo_produtivo(data, periodo):
    linha = periodo.loc[
        (periodo["Desde"] <= data) &
        (periodo["Hasta"] >= data)
    ]

    if not linha.empty:
        ano = linha.iloc[0]["Ano"]
        mes = linha.iloc[0]["Mes"]
        return f"{ano}-{mes:02d}"  # Ex.: 2026-07

    return pd.NA

In [50]:
def tel_number_pdr_val(df, col1):

    import pandas as pd

    df = df.copy()

    nm = 'Telefone_NRZD'

    df[nm] = (
        df[col1]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
        .str.replace(r"[^\d]", "", regex=True)
    )

    # Remove código do país apenas quando fizer sentido
    df[nm] = df[nm].apply(
        lambda x: x[2:] if x.startswith("55") and len(x) >= 12 else x
    )

    # Remove zeros à esquerda DEPOIS
    df[nm] = df[nm].str.replace(r"^0+", "", regex=True)

    return df

In [51]:
def email_pdr_val(df, col1):

    import pandas as pd

    df = df.copy()

    nm = 'Email_NRZD'

    # Normalização
    df[nm] = (
        df[col1]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', '', regex=True)
    )

    # Validação da estrutura
    email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    df['em_validacao'] = (
        df[nm]
        .str.match(email_regex, na=False)
    )

    return df

In [52]:
def carregamento(prefixo, path, hd=None,fluxo_externo=False,):
    import pandas as pd
    import glob
    import os

    arquivos = (
        glob.glob(os.path.join(path, f"{prefixo}*.csv"))
        + glob.glob(os.path.join(path, f"{prefixo}*.xlsx"))
        + glob.glob(os.path.join(path, f"{prefixo}*.xls"))
    )

    dfs = []

    for arq in arquivos:

        extensao = os.path.splitext(arq)[1].lower()

        if extensao == ".csv":
            df = pd.read_csv(arq, sep=';', low_memory=False)

        elif extensao in [".xlsx", ".xls"]:
            if hd is None:
                df = pd.read_excel(arq)
            else:
                df = pd.read_excel(arq, header=hd)

        else:
            continue

        # Sempre guarda a origem
        df['arquivo_origem'] = os.path.basename(arq)

        dfs.append(df)

    if not dfs:
        raise ValueError("Nenhum arquivo encontrado para o prefixo informado.")

    df_final = pd.concat(dfs, ignore_index=True)

    # Remove apenas no fluxo externo
    if fluxo_externo:
        df_final.drop(columns='arquivo_origem', inplace=True)

    return df_final

In [53]:
import os

def pegar_arquivo_recente(pasta, extensao=None, posicao=1):
    """
    Retorna o n-ésimo arquivo mais recente de uma pasta.

    Parâmetros:
        pasta (str): Caminho da pasta.
        extensao (str ou tuple, opcional): Ex.: '.csv', '.xlsx'.
        posicao (int): 1 = mais recente, 2 = penúltimo, 3 = antepenúltimo...

    Retorna:
        str | None: Caminho completo do arquivo ou None se não existir.
    """

    arquivos = [
        os.path.join(pasta, f)
        for f in os.listdir(pasta)
        if os.path.isfile(os.path.join(pasta, f))
    ]

    if extensao:
        arquivos = [
            f for f in arquivos
            if f.lower().endswith(extensao if isinstance(extensao, tuple) else extensao.lower())
        ]

    if len(arquivos) < posicao:
        return None

    arquivos_ordenados = sorted(
        arquivos,
        key=os.path.getctime,
        reverse=True
    )

    return arquivos_ordenados[posicao - 1]

# 2. Carregamento de dados
- datalake: C:\Users\luiz.farias\Numerator International\BKO - projeto-dados-ops;
- pasta_ficha: ...\do_ficha\tc;
- pasta_domicilios: ...\do_domicilios_vivos;
- pasta_all:pre: ...\do_preal_fornecedor\ref_cosolidado;
- pasta_dogacode: ...\do_gacode;
- pasta_bolsa_engaj: ...\do_ficha\tc\output\engj

## 2.1 Atualização de bases de criterio e atos

In [54]:
from wp_rct_atm_atos_tc.main import main

#main(None)

## 2.2 Carregamento de bases principais

In [55]:
# Pasta raiz do projets

# Trocar para domínio interno do pc pessoal
dominio_interno = 'luiz.farias'

# Pasta MASTER
datalake = f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/projeto-dados-ops'

# Pasta Dominio Interno
pasta_ficha = '/do_ficha/tc/input_bruto'
pasta_domicilios = '/do_bases/TopClient'
pasta_all_pre = '/do_preal_fornecedor/ref_cosolidado'
pasta_gacode = '/do_gacode'
pasta_output = '/do_ficha/tc/output_validacao'
pasta_per_produtivo = '/do_calendario_fiscal'
pasta_engj ='/do_ficha/tc/output_engj'

In [56]:
# Ficha de cadastro do panelista
df_periodo= carregamento(    
    path = Path(datalake + pasta_per_produtivo),
    prefixo = 'do_cal'
)

In [57]:
# Ficha de cadastro do panelista
df_ficha = carregamento(    
    path = Path(datalake + pasta_ficha),
    prefixo = 'ficha'
)
df_ficha.loc[df_ficha['arquivo_origem'].str.startswith('ficha-backlog'), 'Status'] = 'BACKLOG'
df_ficha.loc[df_ficha['arquivo_origem'].str.startswith('ficha-eligible'), 'Status'] = 'ENVIADO_WP'

col = ['AVATAR FINALIZADO (BRT)','Gacode','Estado','Entrevistador#1','Entrevistador#2',
       'PanelSmart#1','PanelSmart#1.1','UserPS','Classe','P12a#1','P10a#1', 'P10b#1','P10d_t','P10e#2_t','Status']
df_ficha['UserPS'] = df_ficha['PanelSmart#1'].astype(int) - 550000000


# Lógica: não desejo informar. Retirar se precisar substituir o uso
# df_ficha = df_ficha.loc[~df_ficha.eq('nao_desejo_informar').any(axis=1)].copy()
df_trat = df_ficha[col].copy()


C:\Users\luiz.farias\AppData\Local\Temp\ipykernel_26340\1609117776.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['arquivo_origem'] = os.path.basename(arq)
C:\Users\luiz.farias\AppData\Local\Temp\ipykernel_26340\1609117776.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['arquivo_origem'] = os.path.basename(arq)
C:\Users\luiz.farias\AppData\Local\Temp\ipykernel_26340\1609117776.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has po

In [58]:
df_ficha["data_recebimento"] = pd.to_datetime(
    df_ficha["arquivo_origem"].str.extract(r"-(\d{8})-")[0],
    format="%Y%m%d"
).dt.date

C:\Users\luiz.farias\AppData\Local\Temp\ipykernel_26340\3354969686.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ficha["data_recebimento"] = pd.to_datetime(


In [59]:
df_ficha.columns

Index(['AVATAR FINALIZADO (BRT)', 'idEntrevista', 'Estado', 'Entrevistador#1',
       'Entrevistador#2', 'Entrevistador#3', 'Gacode', 'PanelSmart#1',
       'PanelSmart#1.1', 'Cota_Origem',
       ...
       'SYS_RECORD', 'SYS_TIMEZONE', 'SYS_TIPOENTREVISTADO', 'SYS_TOKEN',
       'SYS_ULTIMAPERGUNTA', 'SYS_USERAGENT', 'arquivo_origem', 'Status',
       'UserPS', 'data_recebimento'],
      dtype='str', length=274)

In [60]:
df_ficha[['arquivo_origem','data_recebimento']]

,arquivo_origem,data_recebimento
0,ficha-total_sessions-20260917.xlsx,NaT
1,ficha-total_sessions-20260917.xlsx,NaT
2,ficha-total_sessions-20260917.xlsx,NaT
3,ficha-total_sessions-20260917.xlsx,NaT
4,ficha-total_sessions-20260917.xlsx,NaT
...,...,...
6609,ficha-total_sessions-20260921.xlsx,NaT
6610,ficha-total_sessions-20260921.xlsx,NaT
6611,ficha-total_sessions-20260921.xlsx,NaT
6612,ficha-total_sessions-20260921.xlsx,NaT


In [61]:
# Domicilios Vivos
df_vivos = carregamento(
    path = Path(datalake + pasta_domicilios),
    prefixo = 'NRPerfil',
    hd = 14
)
df_vivos.rename(columns={'FIPanel1':'Data_Entrada'}, inplace=True)
df_vivos.iddomicilio = df_vivos.iddomicilio.astype(str)
df_vivos_geral = df_vivos[['iddomicilio', 'Data_Entrada','Origen Proveedor']].copy()
df_vivos = df_vivos.loc[df_vivos['Origen Proveedor'] == 'TOP Client'].copy()


# Referência de todos os id's consoidados top client
df_pre_aloc = carregamento(
    path = Path(datalake + pasta_all_pre),
    prefixo = 'consolidado'
)

df_origem_id = df_pre_aloc[['UserPS','Origen_GPM']].drop_duplicates().copy()
df_origem_id.Origen_GPM = df_origem_id.Origen_GPM.replace('22 - Centro Oeste','22 - Centro-Oeste')


In [62]:
df_origem_id.rename(columns={'Origen_GPM':'Pre_Origen'}, inplace=True)
df_origem_id.UserPS = df_origem_id.UserPS.astype(str)



In [63]:
df_gacode = carregamento(
    path = Path(datalake + pasta_gacode),
    prefixo = 'bs'
)

# 3. Análise de Duplicidade & Critério

In [64]:
# Normalização
df_trat = tel_number_pdr_val(df_trat, 'P10e#2_t')
df_trat = email_pdr_val(df_trat, 'P10d_t')

## 3.1 Duplicidades de E-mail e Telefone
1. Contagem de ocorrências;
2. Classificação das duplicidades;

In [65]:
# Contagem de ocorrências
df_dpl_em = df_trat.groupby('Email_NRZD')['PanelSmart#1.1'].count().reset_index().sort_values(by='PanelSmart#1.1',ascending= False)
df_dpl_tl = df_trat.groupby('Telefone_NRZD')['PanelSmart#1.1'].count().reset_index().sort_values(by='PanelSmart#1.1', ascending= False)

# Atribuição de duplicidade
df_dpl_em['Duplicidade_de_Email'] = df_dpl_em['PanelSmart#1.1'] > 1
df_dpl_tl['Duplicidade_de_Telefone'] = df_dpl_tl['PanelSmart#1.1'] > 1

# Cruzamento com base principal
mrg_em = df_trat.merge(df_dpl_em[['Email_NRZD','Duplicidade_de_Email']],on='Email_NRZD', how = 'left')
mrg_tl = mrg_em.merge(df_dpl_tl[['Telefone_NRZD','Duplicidade_de_Telefone']],on='Telefone_NRZD', how = 'left')

# Validação
print('trat: ',df_trat.shape[0])
print('tl: ',df_dpl_tl.shape[0])
print('em: ',df_dpl_em.shape[0])


trat:  6614
tl:  6607
em:  6575


## 3.2 Critério de Atos

In [66]:
path2 = pegar_arquivo_recente(f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/projeto-dados-ops/do_rep7', extensao=None,posicao=1)
df_criterio2 = pd.read_excel(path2)
df_criterio2 = df_criterio2[['UserPS','DiaDeCompra','NumCompras']]

 # Definir os limites das faixas (bins) e os rótulos correspondentes
bins = [0, 4, 9, 14, 19, 24, float('inf')]
labels = ['1 a 4 atos', '5 a 9 atos', '10 a 14 atos', '15 a 19 atos', '20 a 24 atos', '25+ atos']

aux = df_criterio2.groupby(['UserPS']).agg(
    ultima_trasmissao = ('DiaDeCompra', 'max'),
    NumCompras = ('NumCompras','sum')
).reset_index()


aux['FaixaNumCompras'] = pd.cut(aux['NumCompras'], bins=bins, labels=labels, right=True)
mrg_tl['PanelSmart#1'] = mrg_tl['PanelSmart#1'].astype(str)
vl_atos = mrg_tl.merge(aux, left_on = 'PanelSmart#1',right_on='UserPS', how='left')


In [67]:
# Geração base atos recrutamento
 
vl_atos_recrutamento = (
    vl_atos
    .merge(
        df_vivos[['iddomicilio', 'Data_Entrada','Origen Proveedor']], # Ajuste 14/07/2026: Base direto do GPM (todos os vivos)
        left_on='PanelSmart#1',
        right_on='iddomicilio',
        how='outer',
        indicator=True
    )
    .copy()
)

In [68]:
vl_atos_recrutamento['PanelSmart#1'] = vl_atos_recrutamento['PanelSmart#1'].fillna(vl_atos_recrutamento['iddomicilio'])
vl_atos_recrutamento['iddomicilio'] = vl_atos_recrutamento['iddomicilio'].fillna(vl_atos_recrutamento['PanelSmart#1'])


In [69]:
import numpy as np

df = vl_atos_recrutamento

sem_dup_email = df['Duplicidade_de_Email'].eq(False)
sem_dup_tel = df['Duplicidade_de_Telefone'].eq(False)

cond_base = sem_dup_email & sem_dup_tel

condicoes_ok = [
    cond_base & df['FaixaNumCompras'].eq('25+ atos'),
    cond_base & df['NumCompras'].ge(15) & df['Classe'].eq(6),
    cond_base & df['P12a#1'].eq(1) & df['NumCompras'].ge(15),
]

saidas_ok = [
    'OK_25_ATOS',
    'OK_15_COMPRAS_CLASSE_6',
    'OK_P12A_15_COMPRAS',
]

df['saida_decisao'] = np.select(
    condicoes_ok,
    saidas_ok,
    default='AGUARDAR'
)

motivos = []

motivos.append(np.where(df['Duplicidade_de_Email'].eq(True), 'EMAIL_DUPLICADO', ''))
motivos.append(np.where(df['Duplicidade_de_Telefone'].eq(True), 'TELEFONE_DUPLICADO', ''))

cumpre_criterio_qualidade = (
    df['FaixaNumCompras'].eq('25+ atos') |
    (df['NumCompras'].ge(15) & df['Classe'].eq(6)) |
    (df['P12a#1'].eq(1) & df['NumCompras'].ge(15))
)

motivos.append(np.where(~cumpre_criterio_qualidade, 'NAO_CUMPRE_CRITERIO_QUALIDADE', ''))

df['flag_complementar'] = (
    pd.DataFrame(motivos).T
      .apply(lambda x: ' | '.join([i for i in x if i]), axis=1)
)

df.loc[df['saida_decisao'] != 'AGUARDAR', 'flag_complementar'] = ''

# 4.Validação das Origens
- Identificação da origem vs os dados obtidos na ficha;
- Validação de Chave de setor vs Pertencimento à Origem:
    1. Se a chave de setor pertencer às origens pré-determinadas, então será classificado como uma região em que há coleta;
    2. Se a chave de setor não pertencer as origens então, será classificado como uma região em que não há coleta;

- Validação de Origens:
    1. Se a pre-origem for igual a origem obtida pela chave de setor, então será classificada como correta;
    2. Se a pre-origem não for igual àa 2 origens esperadas, então será classificada como incorreta;


In [70]:

# Enriquecimento Regioes
df_regioes_estado = df_gacode[['NM_UF', 'NM_REGIAO']].drop_duplicates()
df_regioes_estado.rename(columns={'NM_UF':'Estado'}, inplace=True)

# Cruzamento para obser a região do estado
vl_atos_recrutamento = vl_atos_recrutamento.merge(df_regioes_estado, on='Estado', how='left')

# Criação da chave de setor para validação do GA Code
vl_atos_recrutamento['chave_setor'] = vl_atos_recrutamento['NM_REGIAO'] + '-' + vl_atos_recrutamento['Estado'] + '-' + vl_atos_recrutamento['Entrevistador#1']

# Normlaização de dados 
import unicodedata

vl_atos_recrutamento["chave_setor"] = vl_atos_recrutamento["chave_setor"].apply(
    lambda x: unicodedata.normalize("NFKD", x)
    .encode("ASCII", "ignore")
    .decode("ASCII")
    if pd.notna(x) else x
)


In [71]:
df_gacode['chave_setor'] = df_gacode['NM_REGIAO'] + '-' + df_gacode['NM_UF'] + '-' + df_gacode['NM_MUN']


df_origem = df_gacode[['NM_REGIAO','chave_setor','Região Kantar PNC 32 REGIONES','Região Expansão 2024','Origem_GPM_32','Origem_GPM_42','GACODE_KANTAR_Nuevo']].drop_duplicates(subset='chave_setor',).copy()
df_origem.rename(columns={
    'Origem_GPM_32':"Origen_Recrutado_32_IHS",
    'Origem_GPM_42':"Origen_Recrutado_42_Expansao",

},inplace=True)
df_origem["chave_setor"] = df_origem["chave_setor"].apply(
    lambda x: unicodedata.normalize("NFKD", x)
    .encode("ASCII", "ignore")
    .decode("ASCII")
    if pd.notna(x) else x
)

In [72]:
df_ids_recrutados_origen_validacao = vl_atos_recrutamento.merge(df_origem, on='chave_setor', how='left')
df_ids_recrutados_origen_validacao = df_ids_recrutados_origen_validacao.merge(df_origem_id, left_on='PanelSmart#1',right_on='UserPS', how='left')

In [73]:
import numpy as np

# Colunas que representam as origens encontradas pela chave de setor
cols_origem = ['Origen_Recrutado_32_IHS', 'Origen_Recrutado_42_Expansao']

# Limpeza (caso o Excel tenha trazido #N/A como texto)
df_ids_recrutados_origen_validacao[cols_origem] = df_ids_recrutados_origen_validacao[cols_origem].replace(['#N/A', '', 'nan', 'None'], np.nan)

# 1. O setor não pertence a nenhum desenho amostral?
cond_setor_nao_contemplado = df_ids_recrutados_origen_validacao[cols_origem].isna().all(axis=1)

# 2. A pré-origem está entre as origens válidas daquele setor?
cond_origem_correta = df_ids_recrutados_origen_validacao[cols_origem].eq(df_ids_recrutados_origen_validacao['Pre_Origen'], axis=0).any(axis=1)

# Resultado final
df_ids_recrutados_origen_validacao['Validacao_Origem'] = np.select(
    [
        cond_setor_nao_contemplado,
        cond_origem_correta
    ],
    [
        'SETOR_NAO_CONTEMPLADO',
        'ORIGEM_CORRETA'
    ],
    default='ORIGEM_INCORRETA'
)

In [74]:
df_ids_recrutados_origen_validacao.columns

Index(['AVATAR FINALIZADO (BRT)', 'Gacode', 'Estado', 'Entrevistador#1',
       'Entrevistador#2', 'PanelSmart#1', 'PanelSmart#1.1', 'UserPS_x',
       'Classe', 'P12a#1', 'P10a#1', 'P10b#1', 'P10d_t', 'P10e#2_t', 'Status',
       'Telefone_NRZD', 'Email_NRZD', 'em_validacao', 'Duplicidade_de_Email',
       'Duplicidade_de_Telefone', 'UserPS_y', 'ultima_trasmissao',
       'NumCompras', 'FaixaNumCompras', 'iddomicilio', 'Data_Entrada',
       'Origen Proveedor', '_merge', 'saida_decisao', 'flag_complementar',
       'NM_REGIAO_x', 'chave_setor', 'NM_REGIAO_y',
       'Região Kantar PNC 32 REGIONES', 'Região Expansão 2024',
       'Origen_Recrutado_32_IHS', 'Origen_Recrutado_42_Expansao',
       'GACODE_KANTAR_Nuevo', 'UserPS', 'Pre_Origen', 'Validacao_Origem'],
      dtype='str')

In [91]:
cond_aprovado = (
        (df_ids_recrutados_origen_validacao['Validacao_Origem'] == 'ORIGEM_CORRETA') &
        (df_ids_recrutados_origen_validacao['saida_decisao'] != 'AGUARDAR') &
        (df_ids_recrutados_origen_validacao['Data_Entrada'].isna())
    )

cond_ajustar_origem = (
        (df_ids_recrutados_origen_validacao['Validacao_Origem'] == 'ORIGEM_INCORRETA') &
        (df_ids_recrutados_origen_validacao['saida_decisao'] != 'AGUARDAR') &
        (df_ids_recrutados_origen_validacao['Data_Entrada'].isna())
    )

cond_setor= (
        df_ids_recrutados_origen_validacao['Validacao_Origem'] == 'SETOR_NAO_CONTEMPLADO'
    )

cond_qualidade = (
        (df_ids_recrutados_origen_validacao['Validacao_Origem'].isin(['ORIGEM_CORRETA','ORIGEM_INCORRETA'])) & 
        (df_ids_recrutados_origen_validacao['NumCompras'] >= 5) &
        (df_ids_recrutados_origen_validacao['Data_Entrada'].isna())
        
    )

cond_gom = (
        ~df_ids_recrutados_origen_validacao['Data_Entrada'].isna()
    )
cond_teste =(
        df_ids_recrutados_origen_validacao['P10d_t'].isin(['teste@gmail.com',
                                                        'admin@gmail.com',
                                                        'ifoodteste7422@gmail.com'])
    )

'''cond_mortalidade =(
        df_ids_recrutados_origen_validacao['FSPanel1'].notna()
    )'''

cond_abaixo_5_atos = (
        ((df_ids_recrutados_origen_validacao['NumCompras'] < 5) | (df_ids_recrutados_origen_validacao['NumCompras'].isna())) &
        (df_ids_recrutados_origen_validacao['Data_Entrada'].isna())
    )

cond_dupli = (
            (df_ids_recrutados_origen_validacao['Duplicidade_de_Email'] == True) | 
            (df_ids_recrutados_origen_validacao['Duplicidade_de_Telefone'] == True)
        )

df_ids_recrutados_origen_validacao['Decisao_Final'] = np.select(
        [
            cond_dupli,
            cond_teste,
            
            cond_aprovado,
            cond_ajustar_origem,
            cond_abaixo_5_atos,
            cond_setor,
            cond_qualidade,
            cond_gom,
        
            
        ],
        [
            'OFF - DUPLICIDADES',
            'OFF - TESTE',
            
            'SUBIR GPM',
            'SUBIR GPM - AJUSTAR ORIGEM',
            'OFF - ABAIXO DE 5 ATOS',
            'OFF - REGIÃO FORA DA COLETA',
            'SUBIR GPM - CRITERIO DE QUALIDADE (>5 ATOS)',
            'GPM - FICHA IMPORTADA',
            
            
        ],
        default=' 1'
        )

In [92]:
df_ids_recrutados_origen_validacao.Decisao_Final.value_counts()

Decisao_Final
OFF - REGIÃO FORA DA COLETA                    5552
GPM - FICHA IMPORTADA                          4037
OFF - ABAIXO DE 5 ATOS                         2271
SUBIR GPM - AJUSTAR ORIGEM                      145
OFF - DUPLICIDADES                               82
SUBIR GPM - CRITERIO DE QUALIDADE (>5 ATOS)      57
SUBIR GPM                                         1
Name: count, dtype: int64

In [76]:
# Validação de GACode
df_ids_recrutados_origen_validacao['validacao_gacode'] = np.select(
    [
        df_ids_recrutados_origen_validacao['GACODE_KANTAR_Nuevo'] == df_ids_recrutados_origen_validacao['Gacode'],
        df_ids_recrutados_origen_validacao['GACODE_KANTAR_Nuevo'] != df_ids_recrutados_origen_validacao['Gacode'],
        df_ids_recrutados_origen_validacao['Gacode'].isna()
                
    ],
    [
        'CORRETO',
        'AJUSTAR MANUAL',
        'FICHA SEM GACODE'
    ],
    default=None
)



In [77]:
import numpy as np

df_ids_recrutados_origen_validacao['Origem_importar'] = np.select(
    [
        df_ids_recrutados_origen_validacao['Pre_Origen'].eq(
            df_ids_recrutados_origen_validacao['Origen_Recrutado_32_IHS']
        ),

        df_ids_recrutados_origen_validacao['Pre_Origen'].eq(
            df_ids_recrutados_origen_validacao['Origen_Recrutado_42_Expansao']
        ),

        df_ids_recrutados_origen_validacao[
            ['Origen_Recrutado_32_IHS',
             'Origen_Recrutado_42_Expansao']
        ].isna().all(axis=1)
    ],
    [
        df_ids_recrutados_origen_validacao['Origen_Recrutado_32_IHS'],
        df_ids_recrutados_origen_validacao['Origen_Recrutado_42_Expansao'],
        'FORA DA COLETA'
    ],
    default=df_ids_recrutados_origen_validacao['Origen_Recrutado_42_Expansao'] # Origem Padrão
)

In [78]:
df_ids_recrutados_origen_validacao['Classific'] = np.select(
    [
        (df_ids_recrutados_origen_validacao['Classe'] == 6) & (df_ids_recrutados_origen_validacao['P12a#1'] == 1),
        (df_ids_recrutados_origen_validacao['Classe'] == 6),  
        (df_ids_recrutados_origen_validacao['P12a#1'] == 1)
                    
    ],
    [
        'MONO_INDIV&NSE_DE',
        'NSE_DE',
        'MONO_INDIV'
    ],
    default= None
)


# 5. Controle de Lotes
- Cada lote deve ser sinalizado pela data de geração do arquivo.

In [314]:
df_ids_recrutados_origen_validacao['Entrevistador#1'].isna().value_counts()


Entrevistador#1
True     5533
False     528
Name: count, dtype: int64

In [315]:
path = pegar_arquivo_recente(f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/projeto-dados-ops{pasta_output}', extensao=None,posicao=1)
df_ls_batch = pd.read_excel(path, sheet_name='Detalhe Validação')
df_ls_batch = df_ls_batch[['PanelSmart#1','Lote','Data_inicio_importacao','data_processamento']]

df_ls_batch['PanelSmart#1'] = df_ls_batch['PanelSmart#1'].astype(str)

df_ls_batch.dropna(subset='PanelSmart#1',inplace=True)

PermissionError: [Errno 13] Permission denied: 'C:/Users/luiz.farias/Numerator International/BKO - Documents/projeto-dados-ops/do_ficha/tc/output_validacao\\PROD_VALIDACAO_FICHA_TC_2026-09.xlsx'

In [ ]:
# Data da base
df_ids_recrutados_origen_validacao["Data_Entrada_GPM"] = pd.to_datetime(
    df_ids_recrutados_origen_validacao["Data_Entrada"],
    format="%d/%m/%Y"
)

# Datas do calendário produtivo
df_periodo["Desde"] = pd.to_datetime(df_periodo["Desde"])
df_periodo["Hasta"] = pd.to_datetime(df_periodo["Hasta"])

In [ ]:
df_ls_batch['PanelSmart#1'] = df_ls_batch['PanelSmart#1'].astype(float).astype(int).astype(str)

In [ ]:
df_ids_recrutados_origen_validacao['mes_produtivo'] = df_ids_recrutados_origen_validacao["Data_Entrada_GPM"].apply(
    lambda x: obter_periodo_produtivo(x, df_periodo)
)

df_rct_batch_vivos_pend_importacao = df_ids_recrutados_origen_validacao.merge(df_ls_batch, on='PanelSmart#1', how='left').copy()

In [ ]:
df_rct_batch_vivos_pend_importacao["dt_ficha"] = (pd.to_datetime(
    df_rct_batch_vivos_pend_importacao["AVATAR FINALIZADO (BRT)"],
    format="%d/%m/%Y %H:%M",
    dayfirst=True
).dt.normalize()
)
df_rct_batch_vivos_pend_importacao['mes_produtivo_ficha'] = df_rct_batch_vivos_pend_importacao["dt_ficha"].apply(
    lambda x: obter_periodo_produtivo(x, df_periodo)
)


In [ ]:
vl = df_vivos[['iddomicilio', 'Data_Entrada','Origen Proveedor']].copy()

In [ ]:
vl['mes_produtivo_ficha'] = vl['Data_Entrada'].apply(
    lambda x: obter_periodo_produtivo(x, df_periodo))

In [ ]:

df_rct_batch_vivos_pend_importacao['mes_produtivo_ficha'] = df_rct_batch_vivos_pend_importacao["dt_ficha"].apply(
    lambda x: obter_periodo_produtivo(x, df_periodo)
) 

In [ ]:
df_rct_batch_vivos_pend_importacao['Entrevistador#1'].isna().value_counts()

Entrevistador#1
True     5533
False     374
Name: count, dtype: int64

In [ ]:
df_ids_recrutados_origen_validacao['Entrevistador#1'].isna().value_counts()


Entrevistador#1
True     5533
False     374
Name: count, dtype: int64

In [ ]:
df_rct_batch_vivos_pend_importacao.mes_produtivo.value_counts()

mes_produtivo
2026-08    2341
2026-07    1541
2026-06    1120
2026-09     558
2026-05     130
2026-04      38
2026-02       1
2026-01       1
Name: count, dtype: int64

In [ ]:
from datetime import datetime

hoje = datetime.now()

periodo_real = obter_periodo_produtivo(hoje, df_periodo)

'''df_rct_batch_vivos_pend_importacao.loc[
    (df_rct_batch_vivos_pend_importacao.mes_produtivo == periodo_real) &
    (df_rct_batch_vivos_pend_importacao.Status.isna()), 'Status'] = 'FICHA_VALIDADA_PERIODO_ANTERIOR'''

df_rct_batch_vivos_pend_importacao = df_rct_batch_vivos_pend_importacao[
    
    (pd.to_datetime(df_rct_batch_vivos_pend_importacao["mes_produtivo"]).dt.month == hoje.month) &
    (pd.to_datetime(df_rct_batch_vivos_pend_importacao["mes_produtivo"]).dt.year == hoje.year)
]

# 6. Output:
***Excel de controle e histórico:** _VALIDACAO_FICHA_TC_16072026.xlsx_* 

1. Overview: Distribuição Das validações (Tabela Dinâmica);
2. Validação: Informação Qualitativa das etapas de validação e tabulação;
3. Ficha (distribuição em sheets pelas tabulações): Dados a serem importação, primeira coluna: **decisão final**;



    





## 6.1 Aplicação Lotes

In [ ]:
from datetime import datetime

data_processamento = datetime.now()


lote = f"GPM_{data_processamento.strftime('%Y%m%d_%H%M%S')}"

mask = (
    df_rct_batch_vivos_pend_importacao["Decisao_Final"].isin([
        "SUBIR GPM",
        "SUBIR GPM - AJUSTAR ORIGEM",
        "SUBIR GPM - CRITERIO DE QUALIDADE (<25 ATOS)"
    ])
    & df_rct_batch_vivos_pend_importacao["Lote"].isna() & df_rct_batch_vivos_pend_importacao["data_processamento"].isna()
)



df_rct_batch_vivos_pend_importacao.loc[mask, "Lote"] = lote
df_rct_batch_vivos_pend_importacao.loc[mask, "Data_inicio_importacao"] = data_processamento
df_rct_batch_vivos_pend_importacao.loc[mask, "data_processamento"] = str(data_processamento)


In [ ]:
df_rct_batch_vivos_pend_importacao[['Lote','data_processamento','Data_inicio_importacao']]

,Lote,data_processamento,Data_inicio_importacao
0,Lote Processo Manual -2026-07-17,NaN,NaT
1,Lote Processo Manual -2026-07-21,NaN,NaT
2,Lote Processo Manual -2026-07-15,NaN,NaT
3,Lote Processo Manual -2026-07-06,NaN,NaT
4,Lote Processo Manual -2026-07-06,NaN,NaT
...,...,...,...
5902,GPM_20260902_100302,2026-09-02 10:03:02.473234,2026-09-02 10:03:02.473
5903,Lote Processo Manual -2026-09-01,NaN,NaT
5904,GPM_20260902_100302,2026-09-02 10:03:02.473234,2026-09-02 10:03:02.473
5905,GPM_20260902_100302,2026-09-02 10:03:02.473234,2026-09-02 10:03:02.473


In [ ]:
df_rct_batch_vivos_pend_importacao['gpm'] = df_rct_batch_vivos_pend_importacao['PanelSmart#1'].isin(df_vivos_geral['iddomicilio'])

In [ ]:
df_rct_batch_vivos_pend_importacao.loc[df_rct_batch_vivos_pend_importacao.mes_produtivo.isna(), 'mes_produtivo'  ] = None

df_rct_batch_vivos_pend_importacao.loc[
    df_rct_batch_vivos_pend_importacao['Lote'].isna(),
    'Lote'
] = (
    'Lote Processo Manual -'
    + df_rct_batch_vivos_pend_importacao.loc[df_rct_batch_vivos_pend_importacao['Lote'].isna(), 'Data_Entrada_GPM'].astype(str)
)



In [ ]:
df_rct_batch_vivos_pend_importacao.loc[(df_rct_batch_vivos_pend_importacao.gpm == True) & (df_rct_batch_vivos_pend_importacao.Decisao_Final == 'SUBIR GPM'), 'Lote'] = 'FLAG_REVISAR_FICHA'

In [ ]:
df_rct_batch_vivos_pend_importacao.loc[(df_rct_batch_vivos_pend_importacao.gpm == True) & (df_rct_batch_vivos_pend_importacao.Decisao_Final == 'SUBIR GPM')][['PanelSmart#1','gpm','Lote','Decisao_Final']]

,PanelSmart#1,gpm,Lote,Decisao_Final
4381,552292396,True,FLAG_REVISAR_FICHA,SUBIR GPM


## 6.4 Agrupamento lotes & Separação de fichas

### 6.4.1 fichas por decisão

In [ ]:
df_ficha['PanelSmart#1'] = df_ficha['PanelSmart#1'].astype(str)

In [ ]:
ficha_out = df_ficha.merge(df_rct_batch_vivos_pend_importacao[['Validacao_Origem','Duplicidade_de_Email','Duplicidade_de_Telefone','flag_complementar',
                                                               'saida_decisao','PanelSmart#1','Pre_Origen','Origen_Recrutado_32_IHS','Origen_Recrutado_42_Expansao',
                                                               'Origem_importar','Decisao_Final','FaixaNumCompras','validacao_gacode','dt_ficha','mes_produtivo_ficha',
                                                               'Data_Entrada_GPM','mes_produtivo',
                                                               'Lote','Data_inicio_importacao','data_processamento','gpm','Duplicidade_de_Email',
                                                                'Duplicidade_de_Telefone', 'Classific', 'chave_setor'  #Base LOTE
                                                               ]], on='PanelSmart#1', how='left')

In [ ]:
ficha_out.columns

Index(['AVATAR FINALIZADO (BRT)', 'idEntrevista', 'Estado', 'Entrevistador#1',
       'Entrevistador#2', 'Entrevistador#3', 'Gacode', 'PanelSmart#1',
       'PanelSmart#1.1', 'Cota_Origem',
       ...
       'Data_Entrada_GPM', 'mes_produtivo', 'Lote', 'Data_inicio_importacao',
       'data_processamento', 'gpm', 'Duplicidade_de_Email',
       'Duplicidade_de_Telefone', 'Classific', 'chave_setor'],
      dtype='str', length=297)

In [ ]:
ficha_out.loc[(ficha_out.Decisao_Final.isin(['SUBIR GPM - CRITERIO DE QUALIDADE (<25 ATOS)',
                                            'SUBIR GPM - AJUSTAR ORIGEM',
                                            'SUBIR GPM'])) & (ficha_out.gpm == True), 'Lote'] = 'NAO_IMPORTAR_' + str(data_processamento)

In [ ]:
ficha_out = ficha_out[[
    'data_processamento','gpm','Status','Lote','Data_inicio_importacao',
    
    'dt_ficha','mes_produtivo_ficha','flag_complementar','Pre_Origen','Origen_Recrutado_32_IHS','Origen_Recrutado_42_Expansao','Origem_importar','validacao_gacode','Validacao_Origem',
    'Data_Entrada_GPM','mes_produtivo',
    'saida_decisao','FaixaNumCompras','Decisao_Final',
    
    'Gacode',
    
    'PanelSmart#1','Estado','Entrevistador#1','Entrevistador#2','Entrevistador#3','Cota_Origem','Cidade_Cota',
    'P2','P7','P8','P9_3#1','P9_3#5','P9_3#6','P9_3#9','P9_3#10','P9_3#12','P9_3#22','P9_3#24','P9_3#25','P9_3#49',
    'P9_3#59','P9_3#62','P9_3#66','P9_3#70','P9_3#71','P9_3#72','P9_3#73','P9_1#68','P9_1#69','P10','P10a#1','P10b#1',
    'P10c','P10d_t','P10e#1_t','P10e#2_t','P10e#3_t','P10f','P10g','P10h','P10i#1','P10k','P12B#1','P12B#2','P12B#3',
    'P12B#4','P12B#5','P12B#6','P12B#7','P12B#8','P12B#9','P12B#10','P12B_1#1','P12B_1#2','P12B_1#3','P12B_1#4','P12B_1#5',
    'P12B_1#6','P12B_1#7','P12B_1#8','P12B_1#9','P12B_1#10','P12C_1','P12C_2','P12C_3','P12C_4','P12C_5','P12C_6','P12C_7',
    'P12C_8','P12C_9','P12C_10','P12D#1','P12D#2','P12D#3','P12D#4','P12D#5','P12D#6','P12D#7','P12D#8','P12D#9','P12D#10',
    'P12E#1','P12E#2','P12E#3','P12E#4','P12E#5','P12E#6','P12E#7','P12E#8','P12E#9','P12E#10','P12G#1_t','P12G#2_t','P12G#3_t',
    'P12G#4_t','P12G#5_t','P12G#6_t','P12G#7_t','P12G#8_t','P12G#9_t','P12G#10_t','P12H#1_t','P12H#2_t','P12H#3_t','P12H#4_t',
    'P12H#5_t','P12H#6_t','P12H#7_t', 'P12H#8_t','P12H#9_t','P12H#10_t','P12I#1','P12I#2','P12I#3','P12I#4','P12I#5','P12I#6',
    'P12I#7','P12I#8','P12I#9','P12I#10','P13b','P13c','P13','P13a',
    'P14','P14a#1','P14a#2','P14a#3','P14a#4','P14a#5','P14a#6','P14a#7','P14b#1','P14b#2','P14b#3','P14b#4','P14b#5','P14b#6','P14b#7',

    'P14c#1','P14c#2','P14c#3','P14c#4','P14c#5',
    'P14c#6','P14c#7','P14d#1','P14d#2','P14d#3','P14d#4','P14d#5','P14d#6','P14d#7','PF14e#1','PF14e#2','PF14e#3','PF14e#4',
    'PF14e#5','PF14e#6','PF14e#7','PF14f#1','PF14f#2','PF14f#3','PF14f#4','PF14f#5','PF14f#6','PF14f#7', 'AVATAR FINALIZADO (BRT)','P12a#1','Duplicidade_de_Email',
    'Duplicidade_de_Telefone', 'Classe', 'Classific', 'chave_setor'


]]

### 6.4.2 Bolsa de engajamento

In [ ]:
leads_bolsa = ficha_out[
     (ficha_out.gpm == False) &      
    ~(ficha_out.Lote.isna()) &
     (ficha_out.Decisao_Final.isin(['SUBIR GPM - CRITERIO DE QUALIDADE (<25 ATOS)',
                                    'SUBIR GPM - AJUSTAR ORIGEM',
                                    'SUBIR GPM']))][[
    "P10b#1",
    'P10a#1',
    'data_processamento',
    'Status',
    'Lote',
    'Data_inicio_importacao',
    'dt_ficha',
    'mes_produtivo_ficha',
    'Data_Entrada_GPM',
    'mes_produtivo',
    'Origem_importar',''
    'FaixaNumCompras',
    'Decisao_Final',
    'PanelSmart#1',
    'P10e#2_t',
    'P10d_t',
    'Estado',
    'Entrevistador#1',
    'Entrevistador#2',
    'gpm',
    ]]

leads_bolsa['Nome'] = leads_bolsa["P10a#1"]+ " "+leads_bolsa["P10b#1"]

leads_bolsa.drop(columns=['P10a#1','P10b#1'],inplace=True)


leads_bolsa.rename(columns={
    'Data_Entrada_GPM':'dt_entrada_GPM',
    'mes_produtivo':'mes_produtivo_GPM',
    'PanelSmart#1':'IdDomicilio',
    'P10e#2_t':'Telefone',
    'P10d_t': 'Email',
    'Entrevistador#1':'Cidade',
    'Entrevistador#2':'Bairro'
},inplace=True)

leads_bolsa= leads_bolsa.loc[leads_bolsa.Lote.str.startswith('GPM')]

In [ ]:
dia = datetime.now().day
dia

3

In [ ]:
nm_arquivo = f'/BOLSA_ENGAJAMENTO_{periodo_real}-{dia}.xlsx'
output_bolsa = datalake + pasta_engj + nm_arquivo

leads_bolsa.to_excel(output_bolsa, index=False)

## 6.2 Detalhe


In [ ]:
from datetime import datetime

ficha_out.rename(columns={'P12a#1':'qtd_indiv(P12a#1)',
                                                    'data_processamento':'data_processamento'}, inplace=True)


ficha_out.loc[~ficha_out['AVATAR FINALIZADO (BRT)'].isna(), 'FICHA_TC'] = 'NO_GPM'
ficha_out.loc[ficha_out['AVATAR FINALIZADO (BRT)'].isna(), 'FICHA_TC'] = 'FA_FICHA'


detalhe = ficha_out[[
    'Lote','Data_inicio_importacao', 'Status',# Base LOTES
    'dt_ficha',
    'mes_produtivo_ficha',
    'Decisao_Final',
    'PanelSmart#1',
    'qtd_indiv(P12a#1)',
    'Classe',
    'Classific',
    'FaixaNumCompras',
    'Data_Entrada_GPM',
    'mes_produtivo',
    'chave_setor',
    'Gacode',
    'validacao_gacode',
    'Pre_Origen',
    'Origen_Recrutado_32_IHS',
    'Origen_Recrutado_42_Expansao',
    'Origem_importar',
    'Duplicidade_de_Email',
    'Duplicidade_de_Telefone', 
    'Validacao_Origem',
    'saida_decisao',
    'flag_complementar',
    'data_processamento'
    
]]

detalhe.loc[detalhe.mes_produtivo == '2026-08'].Status.value_counts(dropna=False)

detalhe.drop_duplicates(inplace=True)


C:\Users\luiz.farias\AppData\Local\Temp\ipykernel_12856\3502906406.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ficha_out.loc[~ficha_out['AVATAR FINALIZADO (BRT)'].isna(), 'FICHA_TC'] = 'NO_GPM'


In [ ]:
overview = (
    detalhe
    .groupby('Decisao_Final')['PanelSmart#1']
    .nunique()
    .reset_index()
).rename(columns={'PanelSmart#1':'iddomicilio'})

ordem = [
    'SUBIR GPM',
    'SUBIR GPM - AJUSTAR ORIGEM',
    'SUBIR GPM - CRITERIO DE QUALIDADE (<25 ATOS)',
    'OFF - REGIÃO FORA DA COLETA',
    'GPM - FICHA IMPORTADA',
    'OFF - DUPLICIDADES',
    'OFF - TESTE'
]

overview['Decisao_Final'] = pd.Categorical(
    overview['Decisao_Final'],
    categories=ordem,
    ordered=True
)

overview = overview.sort_values('Decisao_Final').reset_index(drop=True)

## 6.5 Arquivo Final

In [ ]:
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter

def formatar_sheet(ws):

    # Cores
    cinza = PatternFill("solid", fgColor="CFCFCF")
    azul = PatternFill("solid", fgColor="E3E7FA")
    amarelo = PatternFill("solid", fgColor="FDFFBA")
    vermelho = PatternFill("solid", fgColor="FFD8C2")

    # Cabeçalho
    for cell in ws[1]:
        cell.fill = cinza
        cell.font = Font(color="000000", bold=True)

    # Congela primeira linha
    ws.freeze_panes = "A2"

    # Filtros
    ws.auto_filter.ref = ws.dimensions

    # Ajuste automático da largura
    for col in ws.columns:
        largura = max(len(str(c.value)) if c.value else 0 for c in col) + 2
        ws.column_dimensions[get_column_letter(col[0].column)].width = largura

    # Colunas que deseja destacar
    colunas_azuis = [
        'Validacao_Origem',
        'saida_decisao',
        'Decisao_Final',
        'Pre_Origen',
        '(PNC)Origen_Recrutado_32_IHS',
        '(EXPANSÃO)Origen_Recrutado_42_Expansao',
        'FaixaNumCompras'
    ]

    colunas_amarelas = [
        'Origem_importar',
        'validacao_gacode',
        'Gacode'
        
    ]
    
    colunas_avermelhadas = [
        'flag_complementar'
        
    ]

    for cell in ws[1]:
        if cell.value in colunas_azuis:
            for linha in range(2, ws.max_row + 1):
                ws.cell(row=linha, column=cell.column).fill = azul

        elif cell.value in colunas_amarelas:
            for linha in range(2, ws.max_row + 1):
                ws.cell(row=linha, column=cell.column).fill = amarelo
                
        elif cell.value in colunas_avermelhadas:
            for linha in range(2, ws.max_row + 1):
                ws.cell(row=linha, column=cell.column).fill = vermelho




In [ ]:
from datetime import datetime
from pathlib import Path
import pandas as pd

nm_arquivo = f'/PROD_VALIDACAO_FICHA_TC_{periodo_real}.xlsx'
output = datalake + pasta_output + nm_arquivo

with pd.ExcelWriter(output, engine="openpyxl") as writer:

    # Abas principais
    overview.to_excel(writer, sheet_name="Resumo", index=False)
    detalhe.to_excel(writer, sheet_name="Detalhe Validação", index=False)

    # Cria uma aba para cada Decisao_Final
    for decisao, df in ficha_out.groupby("Decisao_Final", sort=False):

        # Remove caracteres inválidos para nomes de abas
        nome_aba = (
            str(decisao)
            .replace("/", "-")
            .replace("\\", "-")
            .replace("*", "")
            .replace("?", "")
            .replace("[", "")
            .replace("]", "")
            .replace(":", "-")
        )[:31]  # Limite do Excel

        df.to_excel(writer, sheet_name=nome_aba, index=False)
    
    # Ajustando Cores de Colunas
    wb = writer.book

    for ws in wb.worksheets:
        formatar_sheet(ws)
        
        

print(f"Arquivo '{output}' criado com sucesso!")

Arquivo 'C:/Users/luiz.farias/Numerator International/BKO - Documents/projeto-dados-ops/do_ficha/tc/output_validacao/PROD_VALIDACAO_FICHA_TC_2026-09.xlsx' criado com sucesso!
